In [1]:
!pip install -qU crewai langchain-community langchain-huggingface
!pip install -qU transformers accelerate bitsandbytes sentencepiece huggingface_hub pandas tqdm

In [2]:
import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Authenticate
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ Logged in to Hugging Face successfully!")
except Exception as e:
    print("⚠️ Please ensure HF_TOKEN is saved in Kaggle Secrets.")

# 2. Auto-Find and Load Data
file_name = "synthetic_validation_30_fixed.csv"
INPUT_CSV = None

# Search the entire Kaggle directory for the file
for root, dirs, files in os.walk('/kaggle'):
    if file_name in files:
        INPUT_CSV = os.path.join(root, file_name)
        break

if INPUT_CSV:
    print(f"✅ File successfully found at: {INPUT_CSV}")
    df_synth = pd.read_csv(INPUT_CSV, encoding='utf-8')
    print(f"✅ Data loaded successfully! Found {len(df_synth)} rows.")
else:
    print(f"❌ ERROR: Could not find '{file_name}'.")
    print("Troubleshooting: Look at the right sidebar under 'Data' or 'Input'. Make sure the file is actually uploaded there.")

# 3. Context Retrieval Function
def get_extended_context(article, summary, top_n=15):
    sentences = str(article).split('।')
    summary_words = set(str(summary).split())
    scored = []
    for s in sentences:
        overlap = len(set(s.split()).intersection(summary_words))
        scored.append((overlap, s))
    return " । ".join([s[1] for s in sorted(scored, reverse=True, key=lambda x: x[0])[:top_n]]) + " ।"

print("✅ Context functions ready.")

✅ Logged in to Hugging Face successfully!
✅ File successfully found at: /kaggle/input/datasets/subaitabinteibrahim/synthetic-validation-30-fixed/synthetic_validation_30_fixed.csv
✅ Data loaded successfully! Found 30 rows.
✅ Context functions ready.


In [ ]:
import json
import re
import gc
import os
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# The 3 models you are testing for Phase 1
JUDGE_MODELS = [
    "google/gemma-2-9b-it",
    "Qwen/Qwen2.5-7B-Instruct",
    "NousResearch/Meta-Llama-3-8B-Instruct"
]

# 4-bit config to easily fit across the 2x T4 GPUs
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

final_results = {row['article_id']: {'expected_label': row['expected_label']} for _, row in df_synth.iterrows()}
model_accuracies = {}

# --- NATIVE AGENT COMMUNICATION FUNCTION ---
def ask_agent(model, tokenizer, sys_prompt, user_prompt):
    """Handles the communication with the LLM Agent"""
    if "gemma" in model.name_or_path.lower():
        messages = [{"role": "user", "content": f"{sys_prompt}\n\n{user_prompt}"}]
    else:
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=512, 
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Extract only the generated response
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return response


for model_id in JUDGE_MODELS:
    print(f"\n{'='*60}\n STARTING NATIVE MULTI-AGENT PIPELINE: {model_id}\n{'='*60}")
    
    # 1. Load Model and Tokenizer across BOTH GPUs (T4 x2)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="balanced", 
        torch_dtype=torch.float16 
    )

    correct_predictions = 0
    model_col_name = model_id.split('/')[-1].replace('-', '_')
    
    # 2. Process the 30 Samples through the 3 Agents
    for index, row in tqdm(df_synth.iterrows(), total=len(df_synth), desc=f"Auditing via {model_col_name}"):
        art_id = row['article_id']
        expected = row['expected_label']
        bengali_context = get_extended_context(row['article_text'], row['summary_text'])
        
        try:
            # =========================================================
            # AGENT 1: The Fact Extractor
            # =========================================================
            sys_extractor = "You are the Fact Extractor Agent. Extract all atomic claims from the following Bengali summary. Output ONLY a numbered list of English sentences."
            user_extractor = f"Summary: {row['summary_text']}"
            claims_list = ask_agent(model, tokenizer, sys_extractor, user_extractor)

            # =========================================================
            # AGENT 2: The Grounding Verifier
            # =========================================================
            sys_verifier = "You are the Grounding Verifier Agent. Check the extracted English claims against the Bengali article. Mark each as 'Supported' or 'Not Supported'. Be strictly pedantic about names, dates, and numbers."
            user_verifier = f"Bengali Article:\n{bengali_context}\n\nExtracted Claims:\n{claims_list}"
            verification_report = ask_agent(model, tokenizer, sys_verifier, user_verifier)

            # =========================================================
            # AGENT 3: The Judicial Auditor
            # =========================================================
            sys_auditor = """You are the Judicial Auditor Agent. Review the verifier's report. If ANY claim is marked as 'Not Supported', you must output is_faithful: false. Otherwise, output true. 
            Output ONLY valid JSON matching this exact schema: {"is_faithful": boolean, "error_category": "None/Entity/Number/Contradiction/Other"}"""
            user_auditor = f"Verifier Report:\n{verification_report}"
            final_judgment = ask_agent(model, tokenizer, sys_auditor, user_auditor)

            # Parse the final JSON output
            match = re.search(r'\{.*\}', final_judgment.replace('\n', ' '))
            data = json.loads(match.group()) if match else {"is_faithful": False, "error_category": "ParsingError"}
            
            predicted_status = 'Faithful' if data.get('is_faithful') else 'Hallucinated'
            
        except Exception as e:
            predicted_status = 'Error'
            data = {'error_category': 'Execution Failed'}
            print(f"\n[DEBUG] Pipeline Error on row {index}: {str(e)}")

        # Save to our dictionary
        final_results[art_id][f'{model_col_name}_status'] = predicted_status
        final_results[art_id][f'{model_col_name}_error'] = data.get('error_category', 'Error')
        
        if predicted_status == expected:
            correct_predictions += 1

    # Calculate and store accuracy
    acc = correct_predictions / len(df_synth)
    model_accuracies[model_id] = acc
    print(f"\n {model_id} Accuracy: {acc:.2%}")
    
    # 3. CRITICAL: Clear Dual GPU memory for the next model
    print(f" Clearing Dual GPU memory...")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

# --- SAVE FINAL PHASE 1 RESULTS ---
df_results = pd.DataFrame.from_dict(final_results, orient='index').reset_index()
df_results.rename(columns={'index': 'article_id'}, inplace=True)
df_results.to_csv("phase1_model_comparison.csv", index=False)

print("\n" + "="*60)
print(" PHASE 1 COMPLETE: MODEL ACCURACY LEADERBOARD ")
print("="*60)
for m, acc in sorted(model_accuracies.items(), key=lambda item: item[1], reverse=True):
    print(f"{m}: {acc:.2%}")
print("Results saved to 'phase1_model_comparison.csv'. Use the highest scoring model for Phase 2!")

In [3]:
# 1. Force downgrade the datasets library to bypass the recent Hugging Face security block
!pip install -q datasets==2.18.0

import os
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- CONFIGURATION ---
NUM_SAMPLES = 200
GENERATED_OUTPUT_CSV = "mt5_generated_summaries.csv"

# The industry-standard mT5 model fine-tuned for XL-Sum
MT5_MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"

print("="*60)
print(" PHASE 2A: GENERATING SUMMARIES WITH mT5 ")
print("="*60)

# 2. Download the XL-Sum Bengali dataset using the older library version
print("Downloading XL-Sum Bengali dataset...")
# trust_remote_code is required for the xlsum.py script to run
dataset = load_dataset("csebuetnlp/xlsum", name="bengali", split="test", trust_remote_code=True)

# Select only the first 200 samples to save time and resources
subset = dataset.select(range(NUM_SAMPLES))
df_articles = subset.to_pandas()
print(f"Loaded {len(df_articles)} articles.")

# 3. Load the mT5 Model across Dual GPUs (T4 x2)
print(f"Loading {MT5_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MT5_MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MT5_MODEL_NAME,
    device_map="balanced",
    torch_dtype=torch.float16
)

# 4. Generation Loop
generated_data = []

for index, row in tqdm(df_articles.iterrows(), total=len(df_articles), desc="Generating Summaries"):
    article_text = row['text']
    
    # mT5 requires specific formatting and truncation to fit context
    inputs = tokenizer(
        article_text, 
        return_tensors="pt", 
        max_length=1024, 
        truncation=True, 
        padding="max_length"
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=256,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
        
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)
    
    generated_data.append({
        'article_id': row['id'],
        'article_text': article_text,
        'mt5_summary_text': summary
    })

# Save the generated summaries for Phase 2B
df_generated = pd.DataFrame(generated_data)
df_generated.to_csv(GENERATED_OUTPUT_CSV, index=False)
print(f"\n Generation Complete! Saved to {GENERATED_OUTPUT_CSV}")

# 5. CRITICAL: Wipe mT5 from VRAM so Gemma-2 can fit in the next cell
print(" Clearing mT5 from Dual GPUs to free memory...")
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(" GPU memory cleared. You are now ready to run Phase 2B.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 8.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.2.0 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
bigframes 2.35.0 requires rich<14,>=12.4.4, but you have rich 14.3.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gcsfs 2025.3.0 req

Generating train split:   0%|          | 0/8102 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1012 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1012 [00:00<?, ? examples/s]

Loaded 200 articles.
Loading csebuetnlp/mT5_multilingual_XLSum...


...

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generating Summaries:   0%|          | 0/200 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=84) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


 Generation Complete! Saved to mt5_generated_summaries.csv
 Clearing mT5 from Dual GPUs to free memory...
 GPU memory cleared. You are now ready to run Phase 2B.


In [4]:
import json
import re
import gc
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- CONFIGURATION ---
INPUT_CSV = "mt5_generated_summaries.csv"
FINAL_RESULTS_CSV = "final_thesis_results.csv"
WINNING_MODEL = "google/gemma-2-9b-it"

print("="*60)
print(f" PHASE 2B: EVALUATING SUMMARIES WITH {WINNING_MODEL} ")
print("="*60)

# 1. Load the generated summaries
if os.path.exists(INPUT_CSV):
    df_mt5 = pd.read_csv(INPUT_CSV)
    print(f"Loaded {len(df_mt5)} summaries for evaluation.")
else:
    print(f"Error: {INPUT_CSV} not found! Did Phase 2A finish?")
    df_mt5 = pd.DataFrame()

if not df_mt5.empty:
    # 2. Load Gemma-2 in 4-bit across Dual GPUs
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )

    print(f"Loading {WINNING_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(WINNING_MODEL)
    model = AutoModelForCausalLM.from_pretrained(
        WINNING_MODEL,
        quantization_config=bnb_config,
        device_map="balanced", 
        torch_dtype=torch.float16 
    )

    # --- NATIVE AGENT COMMUNICATION FUNCTION ---
    def ask_agent(model, tokenizer, sys_prompt, user_prompt):
        messages = [{"role": "user", "content": f"{sys_prompt}\n\n{user_prompt}"}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs, 
                max_new_tokens=512, 
                do_sample=False, 
                pad_token_id=tokenizer.eos_token_id
            )
        return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # --- CONTEXT RETRIEVAL FUNCTION ---
    def get_extended_context(article, summary, top_n=15):
        sentences = str(article).split('।')
        summary_words = set(str(summary).split())
        scored = []
        for s in sentences:
            overlap = len(set(s.split()).intersection(summary_words))
            scored.append((overlap, s))
        return " । ".join([s[1] for s in sorted(scored, reverse=True, key=lambda x: x[0])[:top_n]]) + " ।"

    phase2_results = []

    # 3. Process the 200 Samples through the 3-Agent Pipeline
    for index, row in tqdm(df_mt5.iterrows(), total=len(df_mt5), desc="Multi-Agent Auditing"):
        art_id = row['article_id']
        article_text = row['article_text']
        summary_text = row['mt5_summary_text']
        
        bengali_context = get_extended_context(article_text, summary_text)
        
        try:
            # Agent 1: Fact Extraction
            sys_extractor = "You are the Fact Extractor Agent. Extract all atomic claims from the following Bengali summary. Output ONLY a numbered list of English sentences."
            claims_list = ask_agent(model, tokenizer, sys_extractor, f"Summary: {summary_text}")

            # Agent 2: Grounding Verification
            sys_verifier = "You are the Grounding Verifier Agent. Check the extracted English claims against the Bengali article. Mark each as 'Supported' or 'Not Supported'. Be strictly pedantic about names, dates, and numbers."
            verification_report = ask_agent(model, tokenizer, sys_verifier, f"Bengali Article:\n{bengali_context}\n\nExtracted Claims:\n{claims_list}")

            # Agent 3: Judicial Auditor
            sys_auditor = """You are the Judicial Auditor Agent. Review the verifier's report. If ANY claim is marked as 'Not Supported', you must output is_faithful: false. Otherwise, output true. 
            Output ONLY valid JSON matching this exact schema: {"is_faithful": boolean, "error_category": "None/Entity/Number/Contradiction/Other"}"""
            final_judgment = ask_agent(model, tokenizer, sys_auditor, f"Verifier Report:\n{verification_report}")

            # Safe JSON parsing
            match = re.search(r'\{.*?\}', final_judgment.replace('\n', ' '))
            if match:
                data = json.loads(match.group())
            else:
                # If JSON fails, assume hallucinated for safety in detection
                data = {"is_faithful": False, "error_category": "ParsingError"}
            
            predicted_status = 'Faithful' if data.get('is_faithful') else 'Hallucinated'
            error_cat = data.get('error_category', 'None')
            
        except Exception as e:
            predicted_status = 'Error'
            error_cat = 'Execution Failed'

        phase2_results.append({
            'article_id': art_id,
            'predicted_status': predicted_status,
            'error_category': error_cat
        })

    # 4. Save Final Statistics
    df_eval = pd.DataFrame(phase2_results)
    df_final = pd.concat([df_mt5, df_eval[['predicted_status', 'error_category']]], axis=1)
    df_final.to_csv(FINAL_RESULTS_CSV, index=False)

    print(f"\n Evaluation Complete! Full data saved to {FINAL_RESULTS_CSV}")
    
    total = len(df_final)
    hallucinated = len(df_final[df_final['predicted_status'] == 'Hallucinated'])
    faithful = len(df_final[df_final['predicted_status'] == 'Faithful'])
    
    print("\n" + "="*40)
    print(" FINAL mT5 HALLUCINATION RESULTS ")
    print("="*40)
    print(f"Total Articles: {total}")
    print(f"Faithful: {faithful} ({(faithful/total)*100:.2f}%)")
    print(f"Hallucinated: {hallucinated} ({(hallucinated/total)*100:.2f}%)")
    print("\nBreakdown of Hallucination Types:")
    print(df_final[df_final['predicted_status'] == 'Hallucinated']['error_category'].value_counts())

    # Final memory cleanup
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()

 PHASE 2B: EVALUATING SUMMARIES WITH google/gemma-2-9b-it 
Loaded 200 summaries for evaluation.
Loading google/gemma-2-9b-it...


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Multi-Agent Auditing:   0%|          | 0/200 [00:00<?, ?it/s]


 Evaluation Complete! Full data saved to final_thesis_results.csv

 FINAL mT5 HALLUCINATION RESULTS 
Total Articles: 200
Faithful: 77 (38.50%)
Hallucinated: 123 (61.50%)

Breakdown of Hallucination Types:
error_category
None             49
Entity           37
Number           15
Claim            13
Other             8
Not Supported     1
Name: count, dtype: int64


In [5]:
import pandas as pd

# Load the results generated in Phase 2B
results_file = "final_thesis_results.csv"

if os.path.exists(results_file):
    df_final = pd.read_csv(results_file)
    
    # Select only the columns requested for GitHub
    # Mapping our internal names to your requested names
    github_df = df_final[['article_id', 'predicted_status', 'error_category']].copy()
    
    # Rename columns to your specific request
    github_df.columns = ['Article ID', 'Status', 'Error Category']
    
    # Save the cleaned version
    github_df.to_csv("github_mt5_results.csv", index=False)
    
    print("Success! 'github_mt5_results.csv' has been generated.")
    print("You can find it in the /kaggle/working directory on the right sidebar.")
    
    # Show the first 5 rows of what you'll be uploading
    print("\nPreview of GitHub Output:")
    print(github_df.head())
else:
    print("Error: Could not find final_thesis_results.csv. Please ensure Phase 2B has finished running.")

Success! 'github_mt5_results.csv' has been generated.
You can find it in the /kaggle/working directory on the right sidebar.

Preview of GitHub Output:
      Article ID        Status Error Category
0  news-45155508      Faithful            NaN
1  news-39127107  Hallucinated          Other
2  news-49141918  Hallucinated         Number
3  news-52808250      Faithful            NaN
4  news-49047323      Faithful            NaN


In [4]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# 1. The exact path you provided for the completed grading sheet
file_path = "/kaggle/input/datasets/subaitabinteibrahim/completed-human-grading/completed_human_grading.csv"

try:
    # 2. Load the completed grading dataset
    df_graded = pd.read_csv(file_path)
    
    # 3. Extract the labels from both raters
    rater1 = df_graded['Rater_1_Label'].tolist()
    rater2 = df_graded['Rater_2_Label'].tolist()
    
    # 4. Calculate Cohen's Kappa
    kappa = cohen_kappa_score(rater1, rater2)
    
    print("\n" + "="*40)
    print(f"📊 Cohen's Kappa Score: {kappa:.4f}")
    print("="*40)
    
    # 5. Academic Interpretation
    print("Interpretation:")
    if kappa <= 0:
        print("No agreement")
    elif kappa <= 0.20:
        print("Slight agreement")
    elif kappa <= 0.40:
        print("Fair agreement")
    elif kappa <= 0.60:
        print("Moderate agreement")
    elif kappa <= 0.80:
        print("Substantial agreement")
    else:
        print("Almost perfect agreement (Excellent for academic validation!)")

except FileNotFoundError:
    print(f"❌ Error: Could not find '{file_path}'.")
    print("Double check that you have added the dataset to your Kaggle notebook by clicking '+ Add Data' on the right sidebar.")
except KeyError as e:
    print(f"❌ Error: Could not find the column {e} in your CSV. Please ensure the columns are named 'Rater_1_Label' and 'Rater_2_Label'.")


📊 Cohen's Kappa Score: 0.8667
Interpretation:
Almost perfect agreement (Excellent for academic validation!)
